In [2]:
!pip install langchain langchain-ollama --quiet

In [17]:
with open("sample.txt", "w") as file:
    file.write("Hello, this is a sample text file.\n")
    file.write("This file was created using Python.\n\n")
    

In [75]:
from langchain.tools import tool
import os

@tool
def read_file(file_path: str) -> str:
    """Read the contents of a file."""
    if not os.path.isfile(file_path):
        return f"The file '{file_path}' was not found."

    with open(file_path, "r") as f:
        content = f.read()

    return f"The contents inside the file are:\n{content}"

@tool
def get_content_length(file_path: str) -> int:
    """
    Returns the number of characters.
    If input_data is a .txt file path, it reads the file and counts the characters.
    Otherwise it treats input_data as plain text.
    """
    
    # Check if it's a txt file path
    if file_path.endswith(".txt") and os.path.isfile(file_path):
        with open(file_path) as f:
            content= len(f.read())
        return f"The file '{file_path}' contains {content} characters."
    elif file_path.endswith(".txt"):
        return f"Error: File '{file_path}' not found."
    else:
        return f"The string '{file_path}' contains {len(file_path)} characters."
   
   
print('Tool name   :', read_file.name)
print('Description :', read_file.description)
print('Schema      :', read_file.args)

@tool
def Count_words(file_path:str)->str:
    """ return the number of words """
    if file_path.endswith(".txt") and os.path.isfile(file_path):
        with open(file_path) as f:
            content= f.read()
        char_length=len(content)
        word_count = len(content.split())
        return f"The file '{file_path}' contains {char_length} characters and {word_count} words"
    elif file_path.endswith(".txt"):
        return f"Error: File '{file_path}' not found."
    else:
        return f"The string '{file_path}' contains {len(file_path.split())} words."
print('Tool name   :', Count_words.name)
print('Description :', Count_words.description)
print('Schema      :', Count_words.args)

    

Tool name   : read_file
Description : Read the contents of a file.
Schema      : {'file_path': {'title': 'File Path', 'type': 'string'}}
Tool name   : Count_words
Description : return the number of words
Schema      : {'file_path': {'title': 'File Path', 'type': 'string'}}


In [76]:
result = read_file.invoke({'file_path':'samp.txt'})
print(result)

Content_length = get_content_length.invoke({'file_path':'Manaswi'})
print(Content_length)

Content_length = get_content_length.invoke({'file_path':'sample.txt'})
print(Content_length)

wordcount = Count_words.invoke({'file_path':'sample.txt'})
print(wordcount)

wordcount = Count_words.invoke({'file_path':'sample, words'})
print(wordcount)

The file 'samp.txt' was not found.
The string 'Manaswi' contains 7 characters.
The file 'sample.txt' contains 416 characters.
The file 'sample.txt' contains 416 characters and 67 words
The string 'sample, words' contains 2 words.


In [77]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.2')
tools = [read_file, get_content_length,Count_words]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke('read the content in sample.txt and get the length of the txt file')
print('Tool calls :', response.tool_calls)
print("LLM bind with tools:",[t.name for t in tools])

Tool calls : [{'name': 'get_content_length', 'args': {'input_data': 'sample.txt'}, 'id': 'de92d865-647e-4d74-9df7-accca9cd648a', 'type': 'tool_call'}]
LLM bind with tools: ['read_file', 'get_content_length', 'Count_words']


In [78]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

tool_map = {'read_file': read_file, 'get_content_length': get_content_length, 'Count_words':Count_words}

SYSTEM_PROMPT = """You are a helpful assistant.
Use the tools to answer the questions."""

def run_agent(USER_MSG: str)->str:
    """ Run the Agent loop for user message"""
# Step 1 — ask the model
    messages = [SystemMessage(content=SYSTEM_PROMPT),
                 HumanMessage(content=USER_MSG)]
    print(f"user message: {USER_MSG}\n")
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            print(f"Agent: {response.content}\n")
            return response.content
        print(response.tool_calls)
# Step 2 — run the tool the model asked for
        for tool_call in response.tool_calls:
            tool_fn = tool_map[tool_call['name']]
            tool_result = tool_fn.invoke(tool_call['args'])
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call['id']))
   



In [79]:

# Test 1 — character count
run_agent("read file sample.txt and give me file content length and number of words?")

user message: read file sample.txt and give me file content length and number of words?

[{'name': 'get_content_length', 'args': {'file_path': 'sample.txt'}, 'id': '0df5ffcc-a6f6-4f05-927c-0d28590ccd96', 'type': 'tool_call'}, {'name': 'Count_words', 'args': {'file_path': 'sample.txt'}, 'id': 'b40e037a-ff4b-4a5a-8d19-bf2d8b0fcf88', 'type': 'tool_call'}]
Agent: **File Content Length and Number of Words**

The content length of the file `sample.txt` is **416 characters**.

The number of words in the file `sample.txt` is **67**.



'**File Content Length and Number of Words**\n\nThe content length of the file `sample.txt` is **416 characters**.\n\nThe number of words in the file `sample.txt` is **67**.'

In [45]:
s="Hello, this is a sample text file\n."
p="This file was created using Python."
i= len(s) +len(p)
print(i)

70


In [69]:
run_agent("""
    1. "How many characters are in sample.txt?"
    2. "What topics are covered in sample.txt?"
    3. "Read sample.txt and give me a one-line summary."
    4. "Read sample.txt and tell me both its content and character count."
    5. "What is in a file called missing.txt?" 
    6.  "get me the length of words in sample.txt file".""")

user message: 
    1. "How many characters are in sample.txt?"
    2. "What topics are covered in sample.txt?"
    3. "Read sample.txt and give me a one-line summary."
    4. "Read sample.txt and tell me both its content and character count."
    5. "What is in a file called missing.txt?" 
    6.  "get me the length of words in sample.txt file".

[{'name': 'get_content_length', 'args': {'file_path': 'sample.txt'}, 'id': '7b3ff982-002e-41e1-a0d0-76f4b079a480', 'type': 'tool_call'}, {'name': 'Count_words', 'args': {'file_path': 'sample.txt'}, 'id': '5c63f5d0-31d6-4a7a-83cc-4aa165826dab', 'type': 'tool_call'}, {'name': 'read_file', 'args': {'file_path': 'sample.txt'}, 'id': '9b93b115-133b-42cb-ba7f-eafb3709d821', 'type': 'tool_call'}, {'name': 'read_file', 'args': {'file_path': 'missing.txt'}, 'id': '92dab97a-75e0-4398-b26d-437f717479bc', 'type': 'tool_call'}, {'name': 'get_content_length', 'args': {'file_path': 'missing.txt'}, 'id': '5a0fda09-d7ce-469c-983c-22fa3c7e7218', 'type': 'tool_c

'Here are the answers to your questions:\n\n1. The file \'sample.txt\' contains 416 characters.\n\n2. The topics covered in sample.txt include electricity, power outage, repair work, transmission line, HECO, Oahu, Maui, Hawaii island, Kauai Island Utility Cooperative and H-3 Freeway.\n\n3. The one-line summary of the contents inside the file \'sample.txt\' is: "Hawaiian Electric said more than 26,000 customers are still without power. They include about 7,000 on Oahu, 4,600 on Maui and 14,500 on Hawaii island."\n\n4. The content of the file \'sample.txt\' is:\n"Hawaiian Electric said more than 26,000 customers are still without power. They include about 7,000 on Oahu, 4,600 on Maui and 14,500 on Hawaii island. At the same time, Kauai Island Utility Cooperative reported 18 active outages across Kauai.\n\nHECO also said the H-3 Freeway was reopened in both directions shortly after 6 p.m. after crews completed repairs to a major transmission line that crosses over traffic."\n\nThe file \'

In [74]:
questions = ["""
    1. "How many characters are in sample.txt?"
    2. "What topics are covered in sample.txt?"
    3. "Read sample.txt and give me a one-line summary."
    4. "Read sample.txt and tell me both its content and character count."
    5. "What is in a file called missing.txt?." """]

for question in questions:
    response = llm_with_tools.invoke(question)
    if response.tool_calls:
        call = response.tool_calls[0]
        print(f'Q: {question}')
        print(f'   -> Model chose tool: {call["name"]} with args {call["args"]}')
    else:
        print(f'Q: {question}')
        print(f'   -> Direct answer: {response.content}')


   

Q: 
    1. "How many characters are in sample.txt?"
    2. "What topics are covered in sample.txt?"
    3. "Read sample.txt and give me a one-line summary."
    4. "Read sample.txt and tell me both its content and character count."
    5. "What is in a file called missing.txt?." 
   -> Model chose tool: get_content_length with args {'file_path': 'sample.txt'}


In [80]:
run_agent("Tell me about the file?")

user message: Tell me about the file?

[{'name': 'read_file', 'args': {'file_path': 'your_file.txt'}, 'id': '3ae03ea1-b07b-4242-a8bb-ca6256b552b1', 'type': 'tool_call'}]
Agent: I don't have any information about a specific file. Can you please provide more context or details about the file you are referring to, such as its location or content? I'll do my best to help.

If you'd like, I can try to find information about a file based on your query. Please provide more details, and I'll use natural language processing (NLP) tools to search for relevant information about the file.

Alternatively, if you have a specific type of file in mind (e.g., Word document, PDF, text file), let me know and I can try to provide more general information about that file format.



"I don't have any information about a specific file. Can you please provide more context or details about the file you are referring to, such as its location or content? I'll do my best to help.\n\nIf you'd like, I can try to find information about a file based on your query. Please provide more details, and I'll use natural language processing (NLP) tools to search for relevant information about the file.\n\nAlternatively, if you have a specific type of file in mind (e.g., Word document, PDF, text file), let me know and I can try to provide more general information about that file format."

1. What is the difference between calling a tool directly (.invoke) and letting the model call it through the agent loop?__
__while we do invoke it calls tools without reasoning. but in agent it searches for reasoning structure to automatically call required tools.__
2. What would happen if you gave the model a very vague question like "Tell me about the file"? Did it still work?
 __The agent asked for clarification when I gave a vague question__
3. What was the most surprising thing the model did automatically?
 __when i have asked a specific question it automatically called the required tools without me telling  what to do__